# Retriever Metrics — DeepEval + RAGAS Side-by-Side

This notebook provides a thorough walkthrough of **retriever evaluation metrics** from both **DeepEval** and **RAGAS**, run on the same RAG pipeline results from Notebook 02.

### What We Cover

| Framework | Metric | What It Measures |
|-----------|--------|------------------|
| DeepEval | ContextualRelevancyMetric | Are retrieved contexts relevant to the query? |
| DeepEval | ContextualPrecisionMetric | Are relevant contexts ranked higher? |
| DeepEval | ContextualRecallMetric | Do contexts cover all claims in the expected output? |
| RAGAS | LLMContextPrecisionWithReference | Are relevant contexts ranked higher? (RAGAS version) |
| RAGAS | LLMContextRecall | Do contexts cover all claims in the reference? |
| RAGAS | ContextEntityRecall | Do contexts capture key entities from the reference? |

-
## 1. Setup & Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

# Load environment
dotenv_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(dotenv_path)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# DeepEval imports
from deepeval.metrics import (
    ContextualRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
)
from deepeval.test_case import LLMTestCase

# RAGAS imports
from ragas import SingleTurnSample, EvaluationDataset, evaluate
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
)
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

print("All imports successful.")

-
## 2. Load Pipeline Results

We load the test case results saved from Notebook 02. If those files are not available, we define the test data inline for self-containedness.

In [ ]:
data_path = os.path.join(os.getcwd(), "data", "pipeline_results.json")

if os.path.exists(data_path):
    with open(data_path) as f:
        pipeline_results = json.load(f)
    print(f"Loaded {len(pipeline_results)} results from {data_path}")
else:
    print("pipeline_results.json not found. Using inline test data.")
    print("Run Notebook 02 first to generate the data, or continue with the inline data below.")
    pipeline_results = None

In [ ]:
# Fallback: inline test data if pipeline_results.json is not available
if pipeline_results is None:
    pipeline_results = [
        {
            "query": "What is the return policy for regular items?",
            "expected_output": "Acme Corp offers a 30-day return policy. Items must be unused, in original packaging, with receipt. Refunds processed in 5-7 business days.",
            "actual_output": "Acme Corp has a 30-day return policy for all products. Items need to be unused and in original packaging with proof of purchase. Refunds are processed within 5-7 business days. Note that shipping costs are the customer's responsibility unless the item was defective.",
            "retrieval_context": [
                "Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. To be eligible for a return, the item must be unused, in its original packaging, and accompanied by the original receipt or proof of purchase. Refunds are processed within 5-7 business days after we receive the returned item. Shipping costs for returns are the responsibility of the customer unless the item was defective or the wrong item was shipped. Items marked as 'Final Sale' cannot be returned or exchanged.",
                "Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days. All electronics must be returned with all original accessories, cables, manuals, and packaging.",
                "Orders can be cancelled within 1 hour of placement by contacting customer support or using the Cancel Order button in your account dashboard."
            ]
        },
        {
            "query": "How long do I have to return electronics?",
            "expected_output": "Electronics have a 15-day return window. They must be returned with all original accessories and packaging. A 15% restocking fee may apply.",
            "actual_output": "Electronic products have a 15-day return window, shorter than the standard 30 days. You must include all original accessories, cables, manuals, and packaging. A 15% restocking fee may apply to opened electronics. Defective electronics can be exchanged within 90 days.",
            "retrieval_context": [
                "Electronic products purchased from Acme Corp have a 15-day return window instead of the standard 30 days. All electronics must be returned with all original accessories, cables, manuals, and packaging. A restocking fee of 15% may apply to opened electronics. Defective electronics can be exchanged for the same model within the first 90 days of purchase at no additional cost.",
                "Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores. To be eligible for a return, the item must be unused, in its original packaging, and accompanied by the original receipt.",
                "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99."
            ]
        },
        {
            "query": "What shipping options are available and how much do they cost?",
            "expected_output": "Standard Shipping (5-7 days, free over $50), Expedited Shipping (2-3 days, $12.99), Overnight Shipping (next day, $24.99 before 2 PM EST).",
            "actual_output": "Acme Corp offers three domestic shipping options: Standard Shipping takes 5-7 business days and is free for orders over $50, Expedited Shipping takes 2-3 business days for $12.99, and Overnight Shipping delivers the next business day for $24.99 if ordered before 2 PM EST.",
            "retrieval_context": [
                "Acme Corp offers several shipping options for domestic orders within the United States. Standard Shipping takes 5-7 business days and is free for orders over $50. Expedited Shipping takes 2-3 business days and costs $12.99. Overnight Shipping is available for $24.99 and delivers the next business day if ordered before 2 PM EST.",
                "Acme Corp ships internationally to over 50 countries. International Standard Shipping takes 10-21 business days and costs vary by destination.",
                "The Acme Rewards program is free to join and earns members 1 point per dollar spent."
            ]
        },
        {
            "query": "Do you ship internationally?",
            "expected_output": "Yes, Acme Corp ships to over 50 countries. Standard takes 10-21 days, Express takes 5-7 days. Customers pay customs duties.",
            "actual_output": "Yes, Acme Corp ships internationally to over 50 countries. International Standard Shipping takes 10-21 business days with costs varying by destination. International Express Shipping takes 5-7 business days. Please note that customs duties, taxes, and import fees are the customer's responsibility.",
            "retrieval_context": [
                "Acme Corp ships internationally to over 50 countries. International Standard Shipping takes 10-21 business days and costs vary by destination. International Express Shipping takes 5-7 business days. All international orders may be subject to customs duties, taxes, and import fees which are the responsibility of the customer.",
                "Acme Corp offers several shipping options for domestic orders within the United States. Standard Shipping takes 5-7 business days and is free for orders over $50.",
                "Acme Corp collects personal information including name, email, shipping address, payment information."
            ]
        },
        {
            "query": "What does the warranty cover?",
            "expected_output": "1-year limited warranty covering defects in materials and workmanship. Does not cover accidents or misuse. Extended plans available.",
            "actual_output": "All Acme Corp branded products come with a 1-year limited warranty that covers defects in materials and workmanship under normal use. The warranty does not cover damage from accidents, misuse, unauthorized modifications, or normal wear and tear. Extended warranty plans for 2 or 3 years are available at purchase time.",
            "retrieval_context": [
                "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship under normal use. The warranty does not cover damage caused by accidents, misuse, unauthorized modifications, or normal wear and tear. Extended warranty plans (2-year and 3-year) are available.",
                "The Acme ErgoDesk Pro is a motorized standing desk priced at $599.99. It comes with a 5-year warranty on the frame and motors.",
                "Acme Corp customer support is available through multiple channels. Phone support is available Monday through Friday."
            ]
        },
        {
            "query": "How can I contact customer support?",
            "expected_output": "Phone: 1-800-ACME-HELP (M-F 8AM-8PM EST). Email: support@acmecorp.com (24-48hr). Live chat: M-Sat 9AM-6PM EST. Help center: help.acmecorp.com.",
            "actual_output": "You can reach Acme Corp support through: Phone at 1-800-ACME-HELP (Monday-Friday 8AM-8PM EST), Email at support@acmecorp.com (24-48 hour response), Live Chat on the website (Monday-Saturday 9AM-6PM EST), or the automated help center at help.acmecorp.com.",
            "retrieval_context": [
                "Acme Corp customer support is available through multiple channels. Phone support is available Monday through Friday, 8 AM to 8 PM EST at 1-800-ACME-HELP. Email support can be reached at support@acmecorp.com with a typical response time of 24-48 hours. Live chat is available Monday through Saturday, 9 AM to 6 PM EST.",
                "Orders can be cancelled within 1 hour of placement by contacting customer support.",
                "Acme Corp collects personal information including name, email, shipping address."
            ]
        },
        {
            "query": "What are the features of the SmartHome Hub?",
            "expected_output": "The SmartHome Hub ($149.99) supports WiFi, Bluetooth, Zigbee, Z-Wave. Has voice control, 5-inch touchscreen, energy monitoring. Works with 10,000+ devices. 2-year warranty.",
            "actual_output": "The Acme SmartHome Hub is priced at $149.99 and supports WiFi, Bluetooth, Zigbee, and Z-Wave protocols, making it compatible with over 10,000 smart home devices. It features built-in voice control, a 5-inch touchscreen display, energy monitoring dashboard, and automated routines. It requires WiFi and is set up via the Acme Home app.",
            "retrieval_context": [
                "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99. It supports WiFi, Bluetooth, Zigbee, and Z-Wave protocols, making it compatible with over 10,000 smart home devices. Features include voice control, a 5-inch touchscreen display, energy monitoring dashboard, and automated routines.",
                "The Acme AirPure Pro is a premium air purifier designed for rooms up to 800 square feet, priced at $299.99.",
                "Acme Corp offers a 30-day return policy on all products."
            ]
        },
        {
            "query": "How much does the AirPure Pro cost and what does it filter?",
            "expected_output": "The AirPure Pro costs $299.99. 4-stage filtration: pre-filter, carbon, True HEPA H13, UV-C. Removes 99.97% of particles 0.3 microns or larger.",
            "actual_output": "The Acme AirPure Pro is priced at $299.99. It features a 4-stage filtration system with a pre-filter, activated carbon filter, True HEPA H13 filter, and UV-C light sanitizer. It removes 99.97% of particles as small as 0.3 microns including dust, pollen, pet dander, mold spores, and smoke.",
            "retrieval_context": [
                "The Acme AirPure Pro is a premium air purifier designed for rooms up to 800 square feet, priced at $299.99. It features a 4-stage filtration system: pre-filter, activated carbon filter, True HEPA H13 filter, and UV-C light sanitizer. The AirPure Pro removes 99.97% of particles as small as 0.3 microns.",
                "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99.",
                "The Acme FitBand Ultra is a fitness tracker priced at $79.99."
            ]
        },
        {
            "query": "What payment methods do you accept?",
            "expected_output": "Visa, MasterCard, AmEx, Discover, PayPal, Apple Pay, Google Pay, Gift Cards. Acme Pay Later for orders over $200.",
            "actual_output": "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. For orders over $200, Acme Pay Later splits into 4 interest-free installments over 6 weeks.",
            "retrieval_context": [
                "Acme Corp accepts the following payment methods: Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay, and Acme Gift Cards. For orders over $200, we also offer Acme Pay Later, a buy-now-pay-later option that splits payments into 4 interest-free installments over 6 weeks.",
                "The Acme Rewards program is free to join and earns members 1 point per dollar spent.",
                "Acme Corp offers a 30-day return policy on all products."
            ]
        },
        {
            "query": "How does the loyalty program work?",
            "expected_output": "Free to join, 1 point per dollar. 100 points = $5 off. Silver (500+ pts): free expedited shipping. Gold (1000+ pts): priority support. Points expire after 12 months inactivity.",
            "actual_output": "The Acme Rewards loyalty program is free. You earn 1 point per dollar spent, and 100 points equals $5 off. Members get early access to sales, 20% birthday discount, and free shipping. Silver tier (500+ points/year) adds free expedited shipping. Gold tier (1000+ points/year) adds priority support. Points expire after 12 months of inactivity.",
            "retrieval_context": [
                "The Acme Rewards program is free to join and earns members 1 point per dollar spent. Points can be redeemed for discounts: 100 points equals $5 off your next purchase. Silver tier (500+ points/year) unlocks free expedited shipping. Gold tier (1000+ points/year) adds priority customer support.",
                "Acme Corp accepts Visa, MasterCard, American Express, Discover, PayPal, Apple Pay, Google Pay.",
                "Acme Corp offers several shipping options for domestic orders."
            ]
        },
        {
            "query": "Can I cancel my order?",
            "expected_output": "Within 1 hour: yes, via support or account dashboard. After 1 hour: may not be possible. Shipped orders must be returned. Custom items cannot be cancelled.",
            "actual_output": "You can cancel an order within 1 hour of placing it through customer support or your account dashboard. After 1 hour, the order enters processing and may not be cancellable. If it has already shipped, you will need to use the return process. Custom items cannot be cancelled once production starts. Refunds go to the original payment method within 3-5 business days.",
            "retrieval_context": [
                "Orders can be cancelled within 1 hour of placement by contacting customer support or using the Cancel Order button in your account dashboard. After 1 hour, orders enter the processing queue and may not be cancellable. If an order has already shipped, it cannot be cancelled and must be returned.",
                "Acme Corp offers a 30-day return policy on all products purchased through our website or retail stores.",
                "Acme Corp customer support is available through multiple channels."
            ]
        },
        {
            "query": "Tell me about the ErgoDesk Pro.",
            "expected_output": "$599.99, 60x30 bamboo top, dual-motor, height 25.5-51 inches, 4 presets, 300 lbs capacity, 5-year frame warranty.",
            "actual_output": "The Acme ErgoDesk Pro is a motorized standing desk at $599.99. It has a 60x30 inch bamboo desktop, dual-motor height adjustment from 25.5 to 51 inches, 4 programmable presets, cable management tray, and anti-collision technology. It supports up to 300 lbs with a 5-year warranty on frame and motors and 2-year warranty on electronics.",
            "retrieval_context": [
                "The Acme ErgoDesk Pro is a motorized standing desk priced at $599.99. It features a 60x30 inch bamboo desktop, dual-motor height adjustment from 25.5 to 51 inches, 4 programmable height presets, built-in cable management tray, and anti-collision technology. The desk supports up to 300 lbs.",
                "All Acme Corp branded products come with a 1-year limited warranty covering defects in materials and workmanship.",
                "The Acme SmartHome Hub is our flagship home automation controller priced at $149.99."
            ]
        }
    ]
    print(f"Using {len(pipeline_results)} inline test cases.")

-
## 3. Create DeepEval Test Cases

DeepEval uses `LLMTestCase` objects. Each test case needs:
- `input` — the user query
- `actual_output` — the generated answer
- `retrieval_context` — list of retrieved context strings
- `expected_output` — the ground truth answer (needed for Precision & Recall)

In [ ]:
test_cases = []

for r in pipeline_results:
    tc = LLMTestCase(
        input=r["query"],
        actual_output=r["actual_output"],
        retrieval_context=r["retrieval_context"],
        expected_output=r["expected_output"],
    )
    test_cases.append(tc)

print(f"Created {len(test_cases)} LLMTestCase objects.")
print(f"\nSample test case:")
print(f"  Input:            {test_cases[0].input[:60]}...")
print(f"  Actual Output:    {test_cases[0].actual_output[:60]}...")
print(f"  Expected Output:  {test_cases[0].expected_output[:60]}...")
print(f"  Contexts:         {len(test_cases[0].retrieval_context)} contexts")

---
### RAGAS Setup

We create RAGAS `SingleTurnSample` objects from the same pipeline results, and configure the evaluator LLM and embeddings.

In [ ]:
# Configure RAGAS evaluator LLM and embeddings
evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))
evaluator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

# Create RAGAS SingleTurnSample objects from pipeline results
ragas_samples = []
for r in pipeline_results:
    sample = SingleTurnSample(
        user_input=r["query"],
        response=r["actual_output"],
        retrieved_contexts=r["retrieval_context"],
        reference=r.get("expected_output", ""),
    )
    ragas_samples.append(sample)

ragas_dataset = EvaluationDataset(samples=ragas_samples)
print(f"Created {len(ragas_samples)} RAGAS samples.")

---
## DeepEval Retriever Metrics

-
## 4. ContextualRelevancyMetric

### What It Measures

**Contextual Relevancy** answers the question: *"Are the retrieved contexts actually relevant to the user's query?"*

The algorithm works as follows:
1. For each retrieved context, an LLM judges whether the context is relevant to the input query.
2. The score is the proportion of contexts that are relevant.

**Score = (number of relevant contexts) / (total number of contexts)**

A score of 1.0 means every retrieved context is relevant. A score of 0.33 means only 1 out of 3 contexts was relevant.

### When to Use
Use this metric to assess whether your retriever is returning noise. If scores are consistently low, your embedding model or chunking strategy may need improvement.

In [ ]:
# Create the metric
contextual_relevancy = ContextualRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {contextual_relevancy.__class__.__name__}")
print(f"Threshold: {contextual_relevancy.threshold}")
print(f"Model:     {contextual_relevancy.model}")

In [ ]:
# Run on all test cases
relevancy_scores = []
relevancy_reasons = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating test case {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    contextual_relevancy.measure(tc)
    relevancy_scores.append(contextual_relevancy.score)
    relevancy_reasons.append(contextual_relevancy.reason)
    print(f"  Score: {contextual_relevancy.score:.4f}")

print(f"\nAverage Contextual Relevancy: {np.mean(relevancy_scores):.4f}")

In [ ]:
# Display scores and reasons
relevancy_df = pd.DataFrame({
    "Query": [tc.input[:50] + "..." for tc in test_cases],
    "Score": relevancy_scores,
    "Passed": [s >= 0.7 for s in relevancy_scores],
    "Reason": [r[:100] + "..." if len(r) > 100 else r for r in relevancy_reasons],
})

print(relevancy_df.to_string(index=False))

### Interpreting Contextual Relevancy Results

- **Score = 1.0:** All retrieved contexts are relevant to the query. Ideal state.
- **Score = 0.67:** Two out of three contexts are relevant. One irrelevant context is being retrieved.
- **Score = 0.33:** Only one out of three contexts is relevant. The retriever is mostly returning noise.
- **Score = 0.0:** None of the retrieved contexts are relevant. Serious retriever problem.

**If scores are low, consider:**
- Improving your chunking strategy (smaller/larger chunks)
- Using a better embedding model
- Adding metadata filters to narrow the search space
- Increasing the number of retrieved chunks before reranking

-
## 5. ContextualPrecisionMetric

### What It Measures

**Contextual Precision** answers: *"Are the relevant contexts ranked higher than the irrelevant ones?"*

This is a ranking-aware metric. Even if you retrieve all the right documents, if they are buried at the bottom of the results (below irrelevant ones), the precision score will be low.

The algorithm:
1. An LLM judges each context as relevant or irrelevant (using the expected output as ground truth).
2. The metric computes a weighted precision score that rewards having relevant contexts at the top.

### When to Use
Use this metric when you want to evaluate not just *what* was retrieved but *how well it was ranked*. This is especially important when you use a reranker — Contextual Precision directly measures reranking quality.

**Note:** This metric requires `expected_output` to judge relevance.

In [ ]:
# Create the metric
contextual_precision = ContextualPrecisionMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {contextual_precision.__class__.__name__}")
print(f"Threshold: {contextual_precision.threshold}")

In [ ]:
# Run on all test cases
precision_scores = []
precision_reasons = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating test case {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    contextual_precision.measure(tc)
    precision_scores.append(contextual_precision.score)
    precision_reasons.append(contextual_precision.reason)
    print(f"  Score: {contextual_precision.score:.4f}")

print(f"\nAverage Contextual Precision: {np.mean(precision_scores):.4f}")

In [ ]:
# Display scores and reasons
precision_df = pd.DataFrame({
    "Query": [tc.input[:50] + "..." for tc in test_cases],
    "Score": precision_scores,
    "Passed": [s >= 0.7 for s in precision_scores],
    "Reason": [r[:100] + "..." if len(r) > 100 else r for r in precision_reasons],
})

print(precision_df.to_string(index=False))

-
## 6. ContextualRecallMetric

### What It Measures

**Contextual Recall** answers: *"Do the retrieved contexts contain all the information needed to produce the expected output?"*

The algorithm:
1. The expected output is broken into individual claims/sentences.
2. An LLM checks whether each claim can be attributed to at least one retrieved context.
3. The score is the proportion of expected claims that are supported by the contexts.

**Score = (number of expected claims supported by contexts) / (total expected claims)**

### When to Use
Use this metric to check if your retriever is finding *all* the relevant information, not just *some* of it. Low recall means the retriever is missing important documents.

**Note:** This metric requires `expected_output` as ground truth.

In [ ]:
# Create the metric
contextual_recall = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o-mini",
)

print(f"Metric:    {contextual_recall.__class__.__name__}")
print(f"Threshold: {contextual_recall.threshold}")

In [ ]:
# Run on all test cases
recall_scores = []
recall_reasons = []

for i, tc in enumerate(test_cases):
    print(f"Evaluating test case {i+1}/{len(test_cases)}: {tc.input[:50]}...")
    contextual_recall.measure(tc)
    recall_scores.append(contextual_recall.score)
    recall_reasons.append(contextual_recall.reason)
    print(f"  Score: {contextual_recall.score:.4f}")

print(f"\nAverage Contextual Recall: {np.mean(recall_scores):.4f}")

In [ ]:
# Display scores and reasons
recall_df = pd.DataFrame({
    "Query": [tc.input[:50] + "..." for tc in test_cases],
    "Score": recall_scores,
    "Passed": [s >= 0.7 for s in recall_scores],
    "Reason": [r[:100] + "..." if len(r) > 100 else r for r in recall_reasons],
})

print(recall_df.to_string(index=False))

### Interpreting Contextual Recall Results

- **Score = 1.0:** All claims in the expected output are supported by retrieved contexts. The retriever found everything needed.
- **Score = 0.75:** Three out of four expected claims are supported. One piece of information is missing from the retrieved contexts.
- **Score = 0.5 or below:** Significant information gaps. The retriever is missing important documents.

**If scores are low, consider:**
- Increasing `top_k` to retrieve more documents
- Improving document chunking (perhaps relevant info is split across chunk boundaries)
- Using hybrid search (combining semantic and keyword search)
- Adding a query expansion step to capture more relevant terms

-
## 7. Compare All Three Metrics

Let's combine all scores into a single DataFrame and visualize them.

In [ ]:
comparison_df = pd.DataFrame({
    "Query": [tc.input[:45] + "..." for tc in test_cases],
    "Relevancy": relevancy_scores,
    "Precision": precision_scores,
    "Recall": recall_scores,
})

# Add average row
avg_row = pd.DataFrame({
    "Query": ["AVERAGE"],
    "Relevancy": [np.mean(relevancy_scores)],
    "Precision": [np.mean(precision_scores)],
    "Recall": [np.mean(recall_scores)],
})
comparison_df = pd.concat([comparison_df, avg_row], ignore_index=True)

print(comparison_df.to_string(index=False))

In [ ]:
# Visualization: grouped bar chart
fig, ax = plt.subplots(figsize=(14, 6))

x = np.arange(len(test_cases))
width = 0.25

bars1 = ax.bar(x - width, relevancy_scores, width, label="Relevancy", color="#4C72B0")
bars2 = ax.bar(x, precision_scores, width, label="Precision", color="#55A868")
bars3 = ax.bar(x + width, recall_scores, width, label="Recall", color="#C44E52")

ax.set_xlabel("Test Case")
ax.set_ylabel("Score")
ax.set_title("Retriever Metrics Comparison Across Test Cases")
ax.set_xticks(x)
ax.set_xticklabels([f"Q{i+1}" for i in range(len(test_cases))], rotation=0)
ax.legend()
ax.set_ylim(0, 1.1)
ax.axhline(y=0.7, color="gray", linestyle="--", alpha=0.5, label="Threshold")

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap style visualization
fig, ax = plt.subplots(figsize=(12, 8))

heatmap_data = np.array([relevancy_scores, precision_scores, recall_scores])
im = ax.imshow(heatmap_data, cmap="RdYlGn", aspect="auto", vmin=0, vmax=1)

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["Relevancy", "Precision", "Recall"])
ax.set_xticks(range(len(test_cases)))
ax.set_xticklabels([f"Q{i+1}" for i in range(len(test_cases))], rotation=45)
ax.set_title("Retriever Metrics Heatmap")

# Add text annotations
for i in range(3):
    for j in range(len(test_cases)):
        text = ax.text(j, i, f"{heatmap_data[i, j]:.2f}",
                       ha="center", va="center", color="black", fontsize=9)

plt.colorbar(im, ax=ax, label="Score")
plt.tight_layout()
plt.show()

---
## RAGAS Retriever Metrics

Now we run the equivalent RAGAS retriever metrics on the **same data**. RAGAS metrics use `SingleTurnSample` objects and are run via `evaluate()`.

## 7. Context Precision

### How Context Precision Works

Context Precision evaluates whether the **relevant chunks appear at the top** of the retrieved results. It measures the signal-to-noise ratio of retrieved contexts with respect to the reference answer.

**Algorithm**:
1. For each retrieved context chunk, the LLM judges whether it is relevant (useful for answering the question given the reference)
2. Precision is calculated at each rank position k: `precision@k = (relevant items in top k) / k`
3. **Score**: Average precision weighted by relevance at each position

High context precision means relevant documents were ranked first. Low context precision means relevant documents are buried among irrelevant ones.

In [ ]:
# Run Context Precision metric
context_precision_metric = LLMContextPrecisionWithReference(llm=evaluator_llm)

context_precision_results = evaluate(
    dataset=ragas_dataset,
    metrics=[context_precision_metric],
)

ctx_prec_df = context_precision_results.to_pandas()
metric_col = ctx_prec_df.columns[-1]
print(f"Context Precision Scores (column: '{metric_col}'):")
print("=" * 60)
for idx, row in ctx_prec_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row[metric_col]:.3f}")
print(f"\nMean Context Precision: {ctx_prec_df[metric_col].mean():.3f}")

## 8. Context Recall

### How Context Recall Works

Context Recall measures whether **all the information needed to answer the question** was actually retrieved. It checks if the ground truth answer can be attributed to the retrieved contexts.

**Algorithm**:
1. The reference (ground-truth) answer is broken into individual sentences/claims
2. For each sentence in the reference, the LLM checks if it can be attributed to any retrieved context
3. **Score**: `Context Recall = (Number of reference sentences attributable to context) / (Total reference sentences)`

A score of 1.0 means the retrieved contexts contain all information in the reference answer. Low recall means important information was not retrieved.

In [ ]:
# Run Context Recall metric
context_recall_metric = LLMContextRecall(llm=evaluator_llm)

context_recall_results = evaluate(
    dataset=ragas_dataset,
    metrics=[context_recall_metric],
)

ctx_recall_df = context_recall_results.to_pandas()
metric_col = ctx_recall_df.columns[-1]
print(f"Context Recall Scores (column: '{metric_col}'):")
print("=" * 60)
for idx, row in ctx_recall_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row[metric_col]:.3f}")
print(f"\nMean Context Recall: {ctx_recall_df[metric_col].mean():.3f}")

## 9. Context Entity Recall

### How Context Entity Recall Works

Context Entity Recall is a complementary metric to regular Context Recall. Instead of checking sentence-level attribution, it focuses on **named entities** (people, organizations, numbers, dates, etc.).

**What it adds over regular recall**:
- Regular recall checks if sentences from the reference can be attributed to context
- Entity recall checks if specific **facts** (entities) mentioned in the reference appear in the retrieved context
- This catches cases where the context paraphrases information but misses specific entities
- It is a non-LLM metric -- purely based on entity extraction and comparison

**Algorithm**:
1. Extract named entities from the reference answer
2. Extract named entities from the retrieved contexts
3. **Score**: `Entity Recall = |entities in reference AND context| / |entities in reference|`

In [ ]:
# Run Context Entity Recall metric
entity_recall_metric = ContextEntityRecall(llm=evaluator_llm)

entity_recall_results = evaluate(
    dataset=ragas_dataset,
    metrics=[entity_recall_metric],
)

entity_recall_df = entity_recall_results.to_pandas()
metric_col = entity_recall_df.columns[-1]
print(f"Context Entity Recall Scores (column: '{metric_col}'):")
print("=" * 60)
for idx, row in entity_recall_df.iterrows():
    query_short = row["user_input"][:55]
    print(f"  [{idx+1:2d}] {query_short:<55} Score: {row[metric_col]:.3f}")
print(f"\nMean Context Entity Recall: {entity_recall_df[metric_col].mean():.3f}")

---
## Side-by-Side: DeepEval vs RAGAS Retriever Metrics

Let's compare the scores from both frameworks on the same test cases.

In [ ]:
# Combine DeepEval and RAGAS retriever scores into one DataFrame
# Note: The RAGAS evaluate() calls above stored results in the results objects.
# Here we build a comparison table.

comparison_data = {
    "Query": [tc.input[:45] + "..." for tc in test_cases],
}

# Add DeepEval scores (already computed above)
comparison_data["DE Relevancy"] = relevancy_scores
comparison_data["DE Precision"] = precision_scores
comparison_data["DE Recall"] = recall_scores

# Add RAGAS scores from the evaluate results
# Use the last column dynamically since RAGAS column names may vary across versions
try:
    cp_df = context_precision_results.to_pandas()
    comparison_data["RAGAS Precision"] = cp_df.iloc[:, -1].tolist()
except:
    print("RAGAS Context Precision scores not available")

try:
    cr_df = context_recall_results.to_pandas()
    comparison_data["RAGAS Recall"] = cr_df.iloc[:, -1].tolist()
except:
    print("RAGAS Context Recall scores not available")

try:
    er_df = entity_recall_results.to_pandas()
    comparison_data["RAGAS Entity Recall"] = er_df.iloc[:, -1].tolist()
except:
    print("RAGAS Entity Recall scores not available")

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))
print(f"\nDeepEval vs RAGAS averages:")
for col in comparison_df.columns[1:]:
    if comparison_df[col].dtype in ['float64', 'int64']:
        print(f"  {col}: {comparison_df[col].mean():.3f}")

---
## Summary

In this notebook we:

1. Loaded RAG pipeline test results from Notebook 02
2. Measured **three DeepEval retriever metrics**: Contextual Relevancy, Precision, and Recall
3. Measured **three RAGAS retriever metrics**: Context Precision, Context Recall, and Entity Recall
4. Compared scores **side-by-side** across both frameworks

### Key Insight
Both frameworks measure similar concepts but use different LLM prompting strategies, which can lead to score differences. Running both gives you a more robust picture.

### Next Steps
Proceed to **Notebook 04** for generator metrics (Faithfulness, Answer Relevancy) from both frameworks.